In [1]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import trackio
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn

keys = 'u', 'v', 'w', 'p_total'

def extract(real, keys):
    with h5py.File("/data1/dataToYouri/couette_dns_ml_f32.h5", 'r') as f:
        data = []
        for k in keys:
            data.append(np.array(f[real][k]))
        return np.stack(data, axis=1)

realizations = ['R13', 'R14', 'R15', 'R18', 'R19', 'R21', 'R22', 'R23', 'R26', 'R28', 'R3', 'R4', 'R5']
samplings = {real: np.s_[:-1:5000 // 300] if real in ["R3", "R4", "R5"] else np.s_[:] for real in realizations} 

raw_d = {real: extract(real, keys)[samplings[real]] for real in realizations}
all_data = np.concatenate(list(raw_d.values()), axis=0)
T, C, X, Y, Z = all_data.shape

# Test and val realizations choosen as they have roughly avg mean
val_realizations = ["R18", "R5"]
test_realizations = ["R3", "R14"]  # ONLY USE FOR FINAL BENCHMARK, ANY DECISION TAKEN AFTER PERFORMANCE IS ESTABLISHED ON THIS DS IS LEAKAGE
train_realizations = [real for real in realizations if real not in val_realizations + test_realizations]
val_npds = np.concatenate([raw_d[r] for r in val_realizations])
train_npds = np.concatenate([raw_d[r] for r in train_realizations])

device = "cuda"

class DS(Dataset):
    def __init__(self, d):
        self.x = torch.from_numpy(d[:, 3:, :, 0, :]).float().to(device)
        self.y = torch.from_numpy(d[:, :3, :, :, :]).float().to(device)
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, i):
        return self.x[i], self.y[i]

normalization_axes = (0, 2, 4,)
all_data_mean = all_data.mean(normalization_axes, keepdims=True)
all_data_std = all_data.std(normalization_axes, keepdims=True) + 1e-10

all_data_mean_gpu = torch.from_numpy(all_data_mean).to(device)
all_data_std_gpu = torch.from_numpy(all_data_std).to(device)

def normalize(data):
    return (data - all_data_mean) / all_data_std

def unnormalize_velocity(data):
    return (data * all_data_std[:, :3]) + all_data_mean[:, :3]

def unnormalize_velocity_gpu(data):
    return (data * all_data_std_gpu[:, :3]) + all_data_mean_gpu[:, :3]

val_ds = DS(normalize(val_npds))
train_ds = DS(normalize(train_npds))

def train(model, opti, loss, ds):
    model.train()
    total_loss = 0
    for x, y in ds:
        opti.zero_grad()
        pred = model(x)
        l = loss(y, pred)
        total_loss += l.item()
        l.backward()
        opti.step()
    return total_loss / len(ds)

def evaluate(model, loss, ds):
    model.eval()
    total_loss = 0
    total_unnormalized_loss = 0
    with torch.no_grad():
        for x, y in ds:
            pred = model(x)
            l = loss(pred, y)
            total_loss += l.item()
            total_unnormalized_loss += total_unnormalized_loss_gpu(pred, y).item()
    return total_loss / len(ds), total_unnormalized_loss / len(ds) 

def predict(model, flat_x_ds):
    model.eval()
    with torch.no_grad():
        pred = model(flat_x_ds)
    return pred

def total_unnormalized_loss_gpu(pred, target):
    unmzed_pred = unnormalize_velocity_gpu(pred)
    unmzed_target = unnormalize_velocity_gpu(target)
    return torch.nn.functional.mse_loss(unmzed_pred, unmzed_target, reduction="mean")

def explained_variance(preds, ys, unmzed_var):
    unormalized_loss = total_unnormalized_loss_gpu(preds, ys)
    explained_var = 1.0 - (unormalized_loss / unmzed_var)
    return explained_var


def train_loop(model, loss, epochs, optim, lr, batch_size, project_name, device):
    
    opti = optim(model.parameters(), lr=lr)
    trackio.init(
        project=project_name,
        config={"epochs": epochs, "learning_rate": lr, "batch_size": batch_size}
    )
    src_test_var = np.var(val_npds)
    for epoch in range(epochs):
        train_error = train(model, opti, loss, train_dl)
        test_error, total_unnormalized_loss = evaluate(model, loss, val_dl)
        
        trackio.log({
            "epoch": epoch,
            "train_error": train_error,
            "test_error": test_error,
            "explained_variance": 1 - (total_unnormalized_loss / src_test_var),
        })
    
    trackio.finish()

batch_size = 4
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)



# train_loop(model, loss, epochs, optim, lr, batch_size, project_name, device)

In [2]:
# define model

epochs = 20
loss = nn.MSELoss(reduction='mean')
lr = 1e-5
optim = torch.optim.Adam
project_name = "linear-ref"

In [3]:
class LinearLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(24 * 24, 3 * 24 * 33 * 24)

    def forward(self, pressure):
        x = pressure.reshape(-1, 24 * 24)
        x = self.linear(x)
        x = x.reshape(-1, 3, 24, 33, 24)
        return x

model = LinearLayer().to(device)

In [4]:
x, y = next(iter(train_dl))
print(x.shape)
pred = model(x)
print(pred.shape == y.shape)
print(pred.shape)
print(y.shape)

torch.Size([4, 1, 24, 24])
True
torch.Size([4, 3, 24, 33, 24])
torch.Size([4, 3, 24, 33, 24])


In [5]:
train_loop(model, loss, epochs, optim, lr, batch_size, project_name, device)

* Trackio project initialized: linear-ref
* Trackio metrics logged to: /home/chancrin/.cache/huggingface/trackio
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: eager-mountain-3


* Run finished. Uploading logs to Trackio (please wait...)


In [6]:
explained_variance(torch.zeros_like(val_dl.dataset.y), val_dl.dataset.y, np.var(val_npds))

tensor(0.5810, device='cuda:0')